# Complete Sentinel-1 C-band versus NISAR L-band FPV benchmark

This notebook runs the complete preregistered experiment: all eligible acquisition-disjoint pairs, full 1–60 m injections in eight directions, 5,000 pair-clustered bootstrap iterations, MD50/80/95, native/matched and harmonized grids, co-pol and dual-pol detectors, temporal-depth matching, and confidence-supported crossover inference.

It does **not** assume that L-band wins. If the data are insufficient or no supported crossover exists, the output says so.

## Before running

1. Attach the private Kaggle Dataset containing this package.
2. Attach the previous FPV handover Dataset containing the three GeoTIFF masks.
3. Add a Kaggle Secret named `EARTHDATA_TOKEN` (preferred), or both `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`.
4. Enable Internet. Select the largest available CPU/RAM runtime; a GPU is not used.

Credentials remain inside Kaggle's secret store and process environment. They are never printed or saved.

In [ ]:
from pathlib import Path
import subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
matches = list(INPUT.rglob('prepare_cross_sensor_data.py'))
if not matches:
    archives = list(INPUT.rglob('Cross_Sensor_FPV_Kaggle_Complete_Experiment*.zip'))
    if not archives:
        visible = '\n'.join(str(p) for p in INPUT.iterdir())
        raise FileNotFoundError(
            'The experiment package is not attached. Use Add Input to attach the private '
            'Kaggle Dataset containing Cross_Sensor_FPV_Kaggle_Complete_Experiment_2026-10-02.zip. '
            f'Currently visible inputs:\n{visible}'
        )
    extracted = Path('/kaggle/working/cross_sensor_package')
    extracted.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(extracted)
    matches = list(extracted.rglob('prepare_cross_sensor_data.py'))
PACKAGE = matches[0].parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE/'requirements.txt')], check=True)
print('Package:', PACKAGE)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
import earthaccess

secrets = UserSecretsClient()
def optional_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

token = optional_secret('EARTHDATA_TOKEN')
if token:
    os.environ['EARTHDATA_TOKEN'] = token
else:
    username = optional_secret('EARTHDATA_USERNAME')
    password = optional_secret('EARTHDATA_PASSWORD')
    if not username or not password:
        raise RuntimeError('Add EARTHDATA_TOKEN or both EARTHDATA_USERNAME and EARTHDATA_PASSWORD as Kaggle Secrets.')
    os.environ['EARTHDATA_USERNAME'] = username
    os.environ['EARTHDATA_PASSWORD'] = password

auth = earthaccess.login(strategy='environment', persist=False)
assert auth.authenticated
print('Earthdata authentication: PASS')

In [ ]:
import json, sys
sys.path.insert(0, str(PACKAGE))
from prepare_cross_sensor_data import prepare
from cross_sensor_inference import run

CONFIG = PACKAGE/'config.json'
# Search inside this package first, preventing duplicate masks from other attached datasets.
INPUT_ROOT = PACKAGE
WORK = Path('/kaggle/working/cross_sensor_fpv_work')
RESULTS = Path('/kaggle/working/cross_sensor_fpv_results')
print(json.dumps(json.loads(CONFIG.read_text()), indent=2)[:12000])

## Stage 1 — Complete acquisition, extraction and pair lock

This searches the full configured archives, streams NISAR GCOV subsets, downloads Sentinel-1 RTC chips, finds the masks, constructs non-overlapping temporal pairs and freezes the chronological 60/40 split. It can take substantial time and disk space.

In [ ]:
audit = prepare(CONFIG, INPUT_ROOT, WORK)
audit

## Stage 2 — Complete inference

Every eligible test pair receives 480 injections (60 magnitudes × eight directions) for every eligible polarization and analysis grid. There is no reduced mode.

In [ ]:
summary = run(CONFIG, WORK/'locked_pairs.csv', RESULTS)
summary

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv(RESULTS/'md_boundaries.csv'))
display(pd.read_csv(RESULTS/'crossover_summary.csv'))
display(pd.read_csv(RESULTS/'power_audit.csv'))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/Cross_Sensor_FPV_Complete_Results', 'zip', RESULTS)
print('Download from the notebook Output panel:', archive)